# Schema Drift in Delta tables

In [0]:
spark.sql("use bronze_wwi")

In [0]:
# File location
file_location = '/Volumes/workspace/default/delta_tables_training/StockItemDelta20250102.csv'
file_type = 'csv'

# CSV options
infer_schema = "false"
first_row_is_header = 'true'
delimiter = ','

In [0]:
# Defining static schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, BooleanType

schema = StructType([
  StructField("StockItemID", IntegerType(), True),
  StructField("StockItemName", StringType(), True),
  StructField("SupplierID", IntegerType(), True),
  StructField("ColorID", IntegerType(), True),
  StructField("UnitPackageID", IntegerType(), True),
  StructField("OuterPackageID", IntegerType(), True),
  StructField("Brand", StringType(), True),
  StructField("Size", StringType(), True),
  StructField("LeadTimeDays", IntegerType(), True),
  StructField("QuantityPerOuter", IntegerType(), True),
  StructField("Barcode", StringType(), True),
  StructField("TaxRate", DoubleType(), True),
  StructField("UnitPrice", DoubleType(), True),
  StructField("RecommendedRetailPrice", DoubleType(), True),
  StructField("TypicalWeightPerUnit", DoubleType(), True),
  StructField("IsChillerStock", BooleanType(), True),
  StructField("ValidFrom", DateType(), True),
  StructField("ValidTo", DateType(), True),
  StructField("update_date", DateType(), True)
])

display(schema)

In [0]:
# Loading the data with the new column
dfadd = spark.read.csv(file_location, header = first_row_is_header, inferSchema = infer_schema, schema = schema, sep = delimiter)
display(dfadd)

In [0]:
# Creating an additional column to store the update date
from pyspark.sql.functions import current_timestamp

df2 = dfadd.withColumn("update_date", current_timestamp())

In [0]:
# Writing and merging the table schema
spark.sql("use bronze_wwi")
df2.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("stockitems")

In [0]:
# Displaying the merged table result
dfbronzeStockItems = spark.sql("select * from bronze_wwi.stockitems")
dfbronzeStockItems.display()

In [0]:
%sql
-- Table history
DESCRIBE HISTORY workspace.bronze_wwi.stockitems